# Phase 1 — Dataset Verification (HARD GATE)

Casting Product Image Data for Quality Inspection (Kaggle, Ravirajsinh Dabhi) — **dataset LOCKED** per design doc v3, section 2.

**Known going in:** the raw distribution has `def_front` (defect) as the RAW MAJORITY (~57%), the opposite of this project's premise. So this notebook does two passes:
1. Verify the RAW pool as downloaded (confirm the known numbers, catch duplicates/corrupt files).
2. Apply the deliberate undersampling construction (keep all `ok_front`, subsample `def_front` down to a stated target ratio).
3. Re-verify the CONSTRUCTED pool — **this is the pool the gate is actually judged against**, and what Phase 2 splits.

**Before running:** unzip the downloaded dataset into the folder pointed at by `config.data.raw_dir`. Confirm the actual class sub-folder names match `config.data.class_folders` — update the YAML if they don't.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.config import load_config
from src.data.verify import (
    list_labeled_images,
    compute_imbalance_report,
    construct_imbalance,
    check_exact_duplicates,
    check_image_properties,
    print_gate_summary,
    lock_dataset,
)

config = load_config("config/config.yaml")
print(f"raw_dir: {config['data']['raw_dir']}")
print(f"class_folders: {config['data']['class_folders']}")

## Pass 1 — raw pool, as downloaded

In [ ]:
# 1.3 — list every labeled image found under raw_dir
df_raw = list_labeled_images(config["data"]["raw_dir"], config["data"]["class_folders"])
print(f"Total images found: {len(df_raw)}")
df_raw.head()

In [ ]:
# Raw imbalance numbers — expect def_front (defect) to be the MAJORITY here,
# matching the ~57%/43% figures already recorded in config.yaml's comments.
imbalance_raw = compute_imbalance_report(df_raw)
imbalance_raw

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.bar(imbalance_raw.class_counts.keys(), imbalance_raw.class_counts.values())
plt.title("Class distribution (RAW pool, before construction)")
plt.ylabel("Image count")
plt.savefig(config["paths"]["figures_dir"] / "phase1_class_distribution_raw.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Duplicate / grouping check on the RAW pool — informs Phase 2's group-aware split
duplicates = check_exact_duplicates(df_raw)
duplicates

In [ ]:
# Image readability / property check on the RAW pool
image_props = check_image_properties(df_raw)
image_props

## Construction step — deliberate imbalance

Per design doc v3 section 2 and `config.data.construct_imbalance`: keep all `ok_front` images, randomly undersample `def_front` down to `config.data.target_majority_minority_ratio` (default 4.0, i.e. ~20% defect prevalence). This decision was stated in the design doc and config **before** this notebook was run — the ratio isn't picked after looking at these results.

In [ ]:
assert config["data"]["construct_imbalance"], "construct_imbalance is False in config — this notebook assumes the constructed-pool path."

df_constructed = construct_imbalance(
    df_raw,
    majority_class="normal",     # ok_front stays at full size
    minority_class="defect",     # def_front gets undersampled
    target_ratio=config["data"]["target_majority_minority_ratio"],
    seed=config["project"]["seed"],
)
print(f"Constructed pool size: {len(df_constructed)}")

## Pass 2 — constructed pool (this is what the gate is judged against)

In [ ]:
imbalance_constructed = compute_imbalance_report(df_constructed)
imbalance_constructed

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(imbalance_constructed.class_counts.keys(), imbalance_constructed.class_counts.values())
plt.title("Class distribution (CONSTRUCTED pool)")
plt.ylabel("Image count")
plt.savefig(config["paths"]["figures_dir"] / "phase1_class_distribution_constructed.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print_gate_summary(imbalance_constructed, duplicates, image_props)
# duplicates/image_props are reused from the raw pool — construction only removes
# images (a subset of the raw pool), so raw findings still apply to what remains.

## Gate decision

Review the printed summary above (constructed pool) against the two gate conditions:
1. Is the minority class (`def_front`) **materially smaller** than the majority (`ok_front`), on these real numbers?
2. Does `def_front` have **enough samples** for a non-trivial train/val/test split (design target: ~785 images, split ~70/15/15 -> ~550/118/118)?

If **yes to both**, run the cell below to lock the dataset — this records the CONSTRUCTED pool's numbers into `config.yaml` (`data.locked`, `data.imbalance_ratio`, `data.minority_prevalence`). If **no to either**, stop here — do not proceed to Phase 2.

In [ ]:
# Gate reviewed and PASSED on the real uploaded data (see outputs/reports/phase1_gate_report.md)
lock_dataset(config, imbalance_constructed, config_path="config/config.yaml")

df_constructed.to_csv(config["data"]["splits_dir"] / "constructed_pool.csv", index=False)
